---

# <span style="color:#1F4E79;">25340 Digital Ocean</span>
### <span style="color:#2E86AB;">Sea Surface Temperature Anomalies and Atmospheric Drivers of the North Atlantic Marine Heatwaves</span>

---

In [5]:
# Import necessary libraries
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import dates as mdates
import copernicusmarine
import warnings
from matplotlib.patches import Patch
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output
from datetime import datetime
import calendar
from pathlib import Path
import os
from getpass import getpass
import cdsapi
import xarray as xr

login: nslimak

password: SzybkieRybki123!

### Sea Surface Temperature (SST)

In [4]:
# Common request settings (smaller area = faster download)
common_request = {
    "dataset_id": "C3S-GLO-SST-L4-REP-OBS-SST",
    "dataset_version": "202506",
    "variables": ["analysed_sst"],
    "minimum_longitude": -80.0,
    "maximum_longitude": 0.0,
    "minimum_latitude": 20.0,
    "maximum_latitude": 65.0,
    "output_directory": "Data",
    "overwrite": True,
}

# Two target periods
periods = [
    {
        "name": "mhw_2016",
        "start": "2015-12-01T00:00:00",
        "end": "2016-04-30T23:59:59",
        "output": "atl_sst_mhw_2016.nc",
    },
    {
        "name": "mhw_2023",
        "start": "2023-05-01T00:00:00",
        "end": "2023-09-30T23:59:59",
        "output": "atl_sst_mhw_2023.nc",
    },
]

for p in periods:
    print(f"Downloading {p['name']}...")
    try:
        copernicusmarine.subset(
            **common_request,
            start_datetime=p["start"],
            end_datetime=p["end"],
            output_filename=p["output"],
        )
        print(f"Done: {p['output']}")
    except Exception as e:
        print(f"Skipped {p['name']}: {e}")

print("Finished all period requests.")

INFO - 2026-03-27T20:42:15Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Copernicus Marine username:Copernicus Marine password:

INFO - 2026-03-27T20:42:31Z - Selected dataset version: "202506"
INFO - 2026-03-27T20:42:31Z - Selected dataset part: "default"
100%|██████████| 341/341 [05:31<00:00,  1.03it/s]
INFO - 2026-03-27T20:48:12Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Done: atl_sst_mhw_2016.nc
Copernicus Marine username:Copernicus Marine password:

INFO - 2026-03-27T20:48:20Z - Selected dataset version: "202506"
INFO - 2026-03-27T20:48:20Z - Selected dataset part: "default"
100%|██████████| 341/341 [05:11<00:00,  1.10it/s]

Done: atl_sst_mhw_2023.nc
Finished all period requests.


In [9]:
# ERA5 derived daily statistics (6-hour frequency) via CDS API

from datetime import datetime
import calendar
from pathlib import Path
import os
from getpass import getpass
import cdsapi
import xarray as xr

# Area of interest: North, West, South, East
area = [61.99, -20.99, 9.01, 12.99]

# Exactly two outputs
periods = [
    {"start": "2015-12-01", "end": "2016-04-30", "output": "era5_tcc_mhw_2016.nc"},
    {"start": "2023-05-01", "end": "2023-09-30", "output": "era5_tcc_mhw_2023.nc"},
]

# CDS key can be overridden by .cdsapirc or CDSAPI_KEY env var
HARDCODED_CDS_KEY = "c3e3c608-3dad-41cd-bc47-98685785ad17"


def iter_months(start_date: datetime, end_date: datetime):
    current = datetime(start_date.year, start_date.month, 1)
    while current <= end_date:
        yield current
        if current.month == 12:
            current = datetime(current.year + 1, 1, 1)
        else:
            current = datetime(current.year, current.month + 1, 1)


def ensure_cds_credentials():
    cds_rc = Path.home() / ".cdsapirc"
    if cds_rc.exists():
        return
    if os.getenv("CDSAPI_KEY") or HARDCODED_CDS_KEY:
        return

    print("CDS credentials not found.")
    print("Paste your CDS API key from your CDS profile page.")
    print("Format: <UID>:<API_KEY>")
    key_value = getpass("CDS key (input hidden): ").strip()
    if not key_value or ":" not in key_value:
        raise RuntimeError("Invalid CDS key format. Expected <UID>:<API_KEY>.")

    cds_rc.write_text(
        "url: https://cds.climate.copernicus.eu/api\n"
        f"key: {key_value}\n",
        encoding="utf-8",
    )
    print(f"Created credentials file at: {cds_rc}")


def build_cds_client():
    ensure_cds_credentials()
    cds_rc = Path.home() / ".cdsapirc"
    if cds_rc.exists():
        return cdsapi.Client()

    cds_url = os.getenv("CDSAPI_URL", "https://cds.climate.copernicus.eu/api")
    cds_key = os.getenv("CDSAPI_KEY", HARDCODED_CDS_KEY)
    if cds_key:
        return cdsapi.Client(url=cds_url, key=cds_key)

    raise RuntimeError("CDS credentials are not available.")


client = build_cds_client()
out_dir = Path("Data")
tmp_dir = out_dir
out_dir.mkdir(parents=True, exist_ok=True)
tmp_dir.mkdir(parents=True, exist_ok=True)

created_files = []

for p in periods:
    start_dt = datetime.fromisoformat(p["start"])
    end_dt = datetime.fromisoformat(p["end"])
    prefix = Path(p["output"]).stem

    monthly_files = []
    for month_dt in iter_months(start_dt, end_dt):
        year = f"{month_dt.year:04d}"
        month = f"{month_dt.month:02d}"
        ndays = calendar.monthrange(month_dt.year, month_dt.month)[1]

        first_day = start_dt.day if (month_dt.year == start_dt.year and month_dt.month == start_dt.month) else 1
        last_day = end_dt.day if (month_dt.year == end_dt.year and month_dt.month == end_dt.month) else ndays

        days = [f"{d:02d}" for d in range(first_day, last_day + 1)]
        month_file = tmp_dir / f"{prefix}_{year}_{month}.nc"
        print(f"Downloading {month_file.name} ...")

        client.retrieve(
            "derived-era5-single-levels-daily-statistics",
            {
                "product_type": "reanalysis",
                "variable": "total_cloud_cover",
                "year": year,
                "month": month,
                "day": days,
                "daily_statistic": "daily_mean",
                "time_zone": "utc+00:00",
                "frequency": "6_hourly",
                "area": area,
                "format": "netcdf",
            },
            str(month_file),
        )
        monthly_files.append(month_file)

    datasets = [xr.open_dataset(fp) for fp in monthly_files]
    try:
        combined = xr.concat(datasets, dim="time").sortby("time")
        target_file = out_dir / p["output"]
        combined.to_netcdf(target_file)
        created_files.append(target_file.name)
        print(f"Created: {target_file.name}")
        combined.close()
    finally:
        for ds in datasets:
            ds.close()
        for fp in monthly_files:
            if fp.exists():
                fp.unlink()

print("Done. Output files:")
for f in created_files:
    print(f" - {f}")

2026-03-27 22:16:57,303 INFO Request ID is 8059d5ee-5dfb-4d80-8f09-3c0432b1912c
2026-03-27 22:16:57,395 INFO status has been updated to accepted
2026-03-27 22:17:11,033 INFO status has been updated to running
2026-03-27 22:17:47,407 INFO status has been updated to successful


2026-03-27 22:17:50,843 INFO Request ID is f9370a5c-e08b-46b6-b288-3bbaff506221
2026-03-27 22:17:50,942 INFO status has been updated to accepted
2026-03-27 22:18:12,445 INFO status has been updated to running
2026-03-27 22:18:23,932 INFO status has been updated to successful


2026-03-27 22:18:26,774 INFO Request ID is eb1aede8-e16e-484d-8c1a-0cfdfa515b5f
2026-03-27 22:18:27,168 INFO status has been updated to accepted
2026-03-27 22:18:40,800 INFO status has been updated to running
2026-03-27 22:18:59,990 INFO status has been updated to successful


2026-03-27 22:19:02,643 INFO Request ID is ef070243-e905-41e0-a2ec-ba1421fd72a7
2026-03-27 22:19:02,741 INFO status has been updated to accepted
2026-03-27 22:19:16,448 INFO status has been updated to running
2026-03-27 22:19:52,789 INFO status has been updated to successful


2026-03-27 22:19:56,103 INFO Request ID is 5e971929-9e11-4562-b56b-43ded37e10b3
2026-03-27 22:19:56,183 INFO status has been updated to accepted
2026-03-27 22:20:09,808 INFO status has been updated to running
2026-03-27 22:20:28,990 INFO status has been updated to successful


...

2026-03-27 22:21:10,294 INFO Request ID is 0ceca5ec-4875-45d6-bf1b-00c430cdd6a3
2026-03-27 22:21:10,393 INFO status has been updated to accepted
2026-03-27 22:21:24,294 INFO status has been updated to running
2026-03-27 22:21:43,451 INFO status has been updated to successful


2026-03-27 22:21:46,459 INFO Request ID is e99ea6cc-11ca-4c5a-a374-6e81230656ba
2026-03-27 22:21:46,559 INFO status has been updated to accepted
2026-03-27 22:22:08,101 INFO status has been updated to running
2026-03-27 22:22:19,581 INFO status has been updated to successful


2026-03-27 22:22:22,835 INFO Request ID is 0da42ddb-a2ce-4d1a-99f2-65c0a90ffede
2026-03-27 22:22:22,932 INFO status has been updated to accepted
2026-03-27 22:22:36,599 INFO status has been updated to running
2026-03-27 22:22:55,756 INFO status has been updated to successful


2026-03-27 22:22:58,655 INFO Request ID is fb1bf866-d13e-4260-b3fd-3d3025d563bd
2026-03-27 22:22:58,762 INFO status has been updated to accepted
2026-03-27 22:23:20,036 INFO status has been updated to running
2026-03-27 22:23:31,525 INFO status has been updated to successful


Created: era5_tcc_mhw_2023.nc
Done. Output files:
 - era5_tcc_mhw_2016.nc
 - era5_tcc_mhw_2023.nc
